# 01. 개별 주식 데이터 수집 및 피처 생성

**목적**: Gu, Kelly, Xiu (2020) 방법론을 개별 주식 유니버스에 적용하기 위한
월별 패널 데이터를 생성한다.

**유니버스**: S&P 500 동적 구성
- Wikipedia 현재 구성 + 변경 히스토리로 역방향 재구성
- 연구 기간(2004~2025) 내 S&P 500에 한 번이라도 편입된 종목 전체 (~500~700개)
- 월별 패널에서 각 날짜의 실제 S&P 500 구성 종목만 포함 → **생존편향 제거**

**수집 기간**: 2004-01-01 ~ 2025-12-31 (**21년**, 활용 변수 고려)
- Walk-forward: IS = 누적 확장(expanding window), OOS = 1개월 → OOS 창 약 180회

**산출 피처 (~50개)**

| 그룹 | 피처 |
|---|---|
| 수익률 | log_ret_1d, simple_ret_1d, excess_ret_1d |
| 모멘텀 | mom_1w/1m/3m/6m/12m/12m_skip_1m, chmom, indmom |
| 변동성/리스크 | vol_20d/60d/252d, beta_252d, idiovol_21d, ivol_63d |
| 유동성 | dollar_vol_21d, amihud_21d, log_mcap, vol_surge |
| 가격 패턴 | high52w_ratio, low52w_ratio, maxret_21d, ma_gap_20_60 |
| 기술적 지표 | rsi_14, bb_pct, intraday_range, autocorr_21d |
| 리스크 조정 수익 | sharpe_21d, sharpe_63d, sortino_63d, ir_63d |
| 분포 | skew_63d, kurt_63d, mdd_252d |
| 거시 민감도 | rate_sensitivity |
| 횡단면(월별) | ret_rank_1m, ret_rank_3m, vol_rank, avg_corr |
| FF 팩터 (거시) | mkt_rf, smb, hml, rmw, cma, rf, mom_factor |

**산출물**: `data/monthly_panel.csv`

---
## Section 0. 설정 및 함수 정의

In [1]:
import bisect
import io
import os
import pickle
import platform
import re
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt

# ── 경로 설정 ─────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
DATA_DIR     = NOTEBOOK_DIR / 'data'
CACHE_DIR    = DATA_DIR / 'cache'
PANELS_DIR   = DATA_DIR / 'panels'
BENCH_DIR    = DATA_DIR / 'benchmarks'

for d in [DATA_DIR, CACHE_DIR, PANELS_DIR, BENCH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 수집 기간 ─────────────────────────────────────────────────────────
PRICE_START = '2004-01-01'
PRICE_END   = '2025-12-31'

# ── GICS 섹터 ↔ 섹터 ETF 매핑 ─────────────────────────────────────────
SECTOR_ETF_MAP = {
    'Energy':                 'XLE',
    'Materials':              'XLB',
    'Industrials':            'XLI',
    'Consumer Discretionary': 'XLY',
    'Consumer Staples':       'XLP',
    'Health Care':            'XLV',
    'Financials':             'XLF',
    'Information Technology': 'XLK',
    'Communication Services': 'XLC',
    'Utilities':              'XLU',
    'Real Estate':            'XLRE',
}

# 다운로드할 보조 데이터 경로
SPY_PATH    = BENCH_DIR / 'SPY.pkl'
DGS10_PATH  = DATA_DIR  / 'dgs10_daily.csv'

# 빌드 시 채워질 전역 변수 (build_panel에서 참조)
SPY_AC      = None  # SPY 조정종가 시리즈
DGS10_DAILY = None  # ^TNX 일별 시리즈

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print(f'DATA_DIR : {DATA_DIR}')
print(f'수집 기간 : {PRICE_START} ~ {PRICE_END}')

DATA_DIR : c:\workspace\camp\project\finance_project\서윤범\Step3_IndividualStocks\data
수집 기간 : 2004-01-01 ~ 2025-12-31


---
## Section 1. 동적 유니버스 구성

**방법**: Wikipedia S&P 500 현재 구성 + 변경 히스토리 → 역방향 재구성
- 현재 구성에서 시작해 과거로 거슬러 가며 추가/제거 이벤트를 역적용
- 연구 기간(2004~2024) 중 S&P 500에 편입된 적 있는 종목 모두 포함
- 월별 패널 구성 시 각 날짜의 실제 멤버만 필터링 → 생존편향 제거

**`get_sp500_members_at(date)`**: 임의 날짜의 S&P 500 구성 종목 반환 (O(log n))

In [2]:
def fetch_sp500_tables():
    url     = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp    = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return pd.read_html(io.StringIO(resp.text))


def parse_current_sp500(table) -> pd.DataFrame:
    df = table.copy()
    df.columns = [c.strip() for c in df.columns]
    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if cl in ('symbol', 'ticker'):      col_map[c] = 'ticker'
        elif cl in ('security', 'company'): col_map[c] = 'company_name'
        elif cl.startswith('gics sector'):  col_map[c] = 'gics_sector'
    df = df.rename(columns=col_map)
    df['ticker'] = df['ticker'].str.replace('.', '-', regex=False).str.strip()
    df['gics_sector'] = df['gics_sector'].replace({
        'Telecommunication Services': 'Communication Services',
        'Healthcare': 'Health Care',
    })
    return df[['ticker', 'company_name', 'gics_sector']].reset_index(drop=True)


def parse_changes_table(raw_table) -> pd.DataFrame:
    """Wikipedia S&P 500 변경 히스토리 테이블 → date / added / removed"""
    df = raw_table.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            '_'.join(str(c) for c in col
                     if 'Unnamed' not in str(c) and str(c).strip()).strip('_')
            for col in df.columns.to_flat_index()
        ]
    df.columns = [c.strip() for c in df.columns]

    date_col    = next((c for c in df.columns if c.lower().startswith('date')), df.columns[0])
    added_col   = next((c for c in df.columns
                        if 'added' in c.lower() and 'ticker' in c.lower()), None)
    removed_col = next((c for c in df.columns
                        if ('removed' in c.lower() or 'deleted' in c.lower())
                        and 'ticker' in c.lower()), None)
    if added_col is None:
        cands = [c for c in df.columns if 'added' in c.lower()]
        added_col = cands[0] if cands else None
    if removed_col is None:
        cands = [c for c in df.columns if 'removed' in c.lower() or 'deleted' in c.lower()]
        removed_col = cands[0] if cands else None

    result = pd.DataFrame({
        'date':    pd.to_datetime(df[date_col], errors='coerce'),
        'added':   df[added_col].astype(str).str.strip()   if added_col   else np.nan,
        'removed': df[removed_col].astype(str).str.strip() if removed_col else np.nan,
    })
    result = result.dropna(subset=['date'])
    for col in ['added', 'removed']:
        result[col] = (result[col]
                       .replace({'nan': np.nan, 'None': np.nan, '—': np.nan,
                                 '-': np.nan, '': np.nan})
                       .str.replace('.', '-', regex=False))
    return result.reset_index(drop=True)


def build_monthly_membership(df_current: pd.DataFrame, df_changes: pd.DataFrame,
                              start: str, end: str) -> dict:
    """
    현재 S&P 500에서 역방향으로 변경사항 적용 → 월말 기준 구성 재구성.
    Returns: {pd.Timestamp(month-end): frozenset of tickers}
    """
    current_set = set(df_current['ticker'].tolist())
    df_ch       = df_changes.sort_values('date', ascending=False).reset_index(drop=True)
    monthly_dates = pd.date_range(start=start, end=end, freq='ME')
    membership  = {}
    ch_idx      = 0

    for target in sorted(monthly_dates, reverse=True):
        while ch_idx < len(df_ch):
            if df_ch.loc[ch_idx, 'date'] <= target:
                break
            added   = df_ch.loc[ch_idx, 'added']
            removed = df_ch.loc[ch_idx, 'removed']
            if pd.notna(added):
                current_set.discard(added)
            if pd.notna(removed):
                current_set.add(removed)
            ch_idx += 1
        membership[target] = frozenset(current_set)

    return membership


# ── 멤버십 히스토리 로드 / 생성 ──────────────────────────────────────
MEMBERSHIP_PATH = DATA_DIR / 'sp500_membership.pkl'

if MEMBERSHIP_PATH.exists():
    with open(MEMBERSHIP_PATH, 'rb') as f:
        sp500_membership = pickle.load(f)
    _membership_dates = sorted(sp500_membership.keys())
    print(f'멤버십 히스토리 로드: {len(sp500_membership)}개 월')
else:
    print('Wikipedia에서 S&P 500 구성 + 변경 히스토리 수집 중...')
    try:
        tables           = fetch_sp500_tables()
        df_current_sp500 = parse_current_sp500(tables[0])
        df_changes_raw   = parse_changes_table(tables[1])
        print(f'  현재 구성: {len(df_current_sp500)}종목 / 변경 이벤트: {len(df_changes_raw)}건')
        sp500_membership  = build_monthly_membership(
            df_current_sp500, df_changes_raw, PRICE_START, PRICE_END
        )
        _membership_dates = sorted(sp500_membership.keys())
        with open(MEMBERSHIP_PATH, 'wb') as f:
            pickle.dump(sp500_membership, f)
        print(f'  멤버십 생성 완료: {len(sp500_membership)}개 월')
    except Exception as e:
        print(f'  멤버십 구성 실패: {e}')
        print('  → 현재 S&P 500 정적 유니버스로 대체')
        sp500_membership  = None
        _membership_dates = []


# ── 전체 유니버스 (연구 기간 중 등장한 모든 종목) ─────────────────────
OUR_UNIVERSE_PATH = DATA_DIR / 'universe.csv'

if OUR_UNIVERSE_PATH.exists():
    df_universe = pd.read_csv(OUR_UNIVERSE_PATH)
    print(f'기존 universe.csv 로드: {len(df_universe)}종목')
else:
    tables_wiki   = fetch_sp500_tables()
    df_current    = parse_current_sp500(tables_wiki[0])
    sector_lookup = dict(zip(df_current['ticker'], df_current['gics_sector']))

    if sp500_membership is not None:
        all_hist = set()
        for members in sp500_membership.values():
            all_hist.update(members)
        rows = [{'ticker': t, 'gics_sector': sector_lookup[t]}
                for t in sorted(all_hist) if t in sector_lookup]
    else:
        rows = df_current[['ticker', 'gics_sector']].to_dict('records')

    df_universe = pd.DataFrame(rows)
    df_universe.to_csv(OUR_UNIVERSE_PATH, index=False, encoding='utf-8-sig')
    print(f'유니버스 저장: {len(df_universe)}종목')

tickers    = df_universe['ticker'].tolist()
sector_map = dict(zip(df_universe['ticker'], df_universe['gics_sector']))


# ── 날짜별 멤버 조회 (O(log n)) ───────────────────────────────────────
def get_sp500_members_at(date: pd.Timestamp) -> set:
    if not _membership_dates:
        return set(tickers)
    idx = bisect.bisect_right(_membership_dates, date) - 1
    if idx < 0:
        return set()
    return set(sp500_membership[_membership_dates[idx]])


print(f'\n유니버스: {len(tickers)}종목')
print(f'  2010-01-31 멤버 수: {len(get_sp500_members_at(pd.Timestamp("2010-01-31")))}')
print(f'  2020-01-31 멤버 수: {len(get_sp500_members_at(pd.Timestamp("2020-01-31")))}')
print(df_universe.groupby('gics_sector').size().to_string())

멤버십 히스토리 로드: 264개 월
기존 universe.csv 로드: 498종목

유니버스: 498종목
  2010-01-31 멤버 수: 510
  2020-01-31 멤버 수: 506
gics_sector
Communication Services    22
Consumer Discretionary    48
Consumer Staples          35
Energy                    22
Financials                76
Health Care               58
Industrials               78
Information Technology    71
Materials                 26
Real Estate               31
Utilities                 31


---
## Section 2. 개별 종목 가격 수집 (2004~2025)

In [3]:
def safe_download(ticker, start=PRICE_START, end=PRICE_END, max_retry=3):
    for attempt in range(max_retry + 1):
        try:
            df = yf.Ticker(ticker).history(
                start=start, end=end,
                auto_adjust=False, actions=True,
            )
            if df.empty:
                return None
            df.index = df.index.tz_localize(None) if df.index.tz else df.index
            df.index.name = 'date'
            return df
        except Exception:
            if attempt < max_retry:
                time.sleep(2 ** attempt)
    return None


ok, fail = [], []
for i, ticker in enumerate(tickers, 1):
    cache_path = CACHE_DIR / f'{ticker}.pkl'
    if cache_path.exists():
        ok.append(ticker)
        if i % 30 == 0:
            print(f'  [{i:3d}/{len(tickers)}] 캐시 스킵 중...')
        continue

    df = safe_download(ticker)
    if df is None:
        fail.append(ticker)
        print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → FAIL')
    else:
        with open(cache_path, 'wb') as f:
            pickle.dump(df, f)
        ok.append(ticker)
        if i % 10 == 0 or i == len(tickers):
            print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → OK ({len(df)}행)')
    time.sleep(0.3)

print(f'\n완료: 성공 {len(ok)}개 / 실패 {len(fail)}개')
if fail:
    print(f'실패: {fail}')

  [ 30/498] 캐시 스킵 중...
  [ 60/498] 캐시 스킵 중...
  [ 90/498] 캐시 스킵 중...
  [120/498] 캐시 스킵 중...
  [150/498] 캐시 스킵 중...
  [180/498] 캐시 스킵 중...
  [210/498] 캐시 스킵 중...
  [240/498] 캐시 스킵 중...
  [270/498] 캐시 스킵 중...
  [300/498] 캐시 스킵 중...
  [330/498] 캐시 스킵 중...
  [360/498] 캐시 스킵 중...
  [390/498] 캐시 스킵 중...
  [420/498] 캐시 스킵 중...
  [450/498] 캐시 스킵 중...
  [480/498] 캐시 스킵 중...

완료: 성공 498개 / 실패 0개


---
## Section 3. Fama-French 5 Factor + MOM 수집

In [4]:
FF_SAVE = DATA_DIR / 'ff_factors.csv'

if FF_SAVE.exists():
    df_ff = pd.read_csv(FF_SAVE, index_col='date', parse_dates=True)
    print(f'기존 ff_factors.csv 로드 ({len(df_ff)}행)')
else:
    def download_ff_zip(url):
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            raw = zf.read(zf.namelist()[0]).decode('utf-8', errors='ignore')
        lines = raw.splitlines()
        start_idx = next(i for i, ln in enumerate(lines) if re.match(r'^\s*\d{8}\s*,', ln))
        end_idx = next(
            (i for i in range(start_idx, len(lines)) if not re.match(r'^\s*\d{8}\s*,', lines[i])),
            len(lines)
        )
        block = '\n'.join(lines[start_idx - 1: end_idx])
        df = pd.read_csv(io.StringIO(block))
        df.columns = [c.strip() for c in df.columns]
        date_col = df.columns[0]
        df[date_col] = pd.to_datetime(df[date_col].astype(int).astype(str), format='%Y%m%d')
        return df.rename(columns={date_col: 'date'}).set_index('date').astype(float) / 100.0

    FF5_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip'
    MOM_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip'

    print('FF5 다운로드 중...')
    df_ff5 = download_ff_zip(FF5_URL)
    df_ff5.columns = [c.strip() for c in df_ff5.columns]
    df_ff5 = df_ff5.rename(columns={'Mkt-RF': 'mkt_rf', 'SMB': 'smb', 'HML': 'hml',
                                    'RMW': 'rmw', 'CMA': 'cma', 'RF': 'rf'})
    print('MOM 다운로드 중...')
    df_mom = download_ff_zip(MOM_URL)
    df_mom.columns = [c.strip() for c in df_mom.columns]
    df_mom = df_mom.rename(columns={df_mom.columns[0]: 'mom_factor'})

    df_ff = df_ff5.join(df_mom[['mom_factor']], how='inner')
    df_ff = df_ff[(df_ff.index >= pd.Timestamp(PRICE_START)) &
                  (df_ff.index <= pd.Timestamp(PRICE_END))]
    df_ff.to_csv(FF_SAVE, encoding='utf-8-sig')
    print(f'FF 팩터 저장: {len(df_ff)}행')

print(f'FF 팩터 기간: {df_ff.index[0].date()} ~ {df_ff.index[-1].date()}')
print(f'컬럼: {df_ff.columns.tolist()}')

기존 ff_factors.csv 로드 (5535행)
FF 팩터 기간: 2004-01-02 ~ 2025-12-31
컬럼: ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor']


---
## Section 4. 섹터 ETF 수집 (산업 모멘텀 계산용)

In [5]:
etf_symbols = list(SECTOR_ETF_MAP.values())  # XLE, XLB, ...
bench_data  = {}  # symbol → DataFrame

for sym in etf_symbols:
    cache_path = BENCH_DIR / f'{sym}.pkl'
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            bench_data[sym] = pickle.load(f)
        print(f'  {sym}: 캐시 로드 ({len(bench_data[sym])}행)')
        continue

    df = safe_download(sym)
    if df is None:
        print(f'  {sym}: FAIL')
    else:
        bench_data[sym] = df
        with open(cache_path, 'wb') as f:
            pickle.dump(df, f)
        print(f'  {sym}: OK ({len(df)}행)')
    time.sleep(0.3)

# 섹터 ETF 조정종가 → adj_close 시리즈로 추출
etf_adj_close = {}
for sym, df in bench_data.items():
    cols_lower = {c.lower().replace(' ', '_'): c for c in df.columns}
    ac_col = cols_lower.get('adj_close') or cols_lower.get('adjclose') or cols_lower.get('close')
    if ac_col:
        s = df[ac_col].copy()
        s.index = s.index.tz_localize(None) if s.index.tz else s.index
        etf_adj_close[sym] = s

# SECTOR → indmom(12개월 수익률) 계산
# 각 섹터 ETF의 252일 rolling return
indmom_by_sector = {}
for sector, sym in SECTOR_ETF_MAP.items():
    if sym in etf_adj_close:
        s = etf_adj_close[sym]
        indmom_by_sector[sector] = (1 + s.pct_change()).rolling(252).apply(np.prod, raw=True) - 1

print(f'\n섹터 ETF 로드 완료: {len(etf_adj_close)}개')
print(f'indmom 계산 완료: {len(indmom_by_sector)}개 섹터')

  XLE: 캐시 로드 (5534행)
  XLB: 캐시 로드 (5534행)
  XLI: 캐시 로드 (5534행)
  XLY: 캐시 로드 (5534행)
  XLP: 캐시 로드 (5534행)
  XLV: 캐시 로드 (5534행)
  XLF: 캐시 로드 (5534행)
  XLK: 캐시 로드 (5534행)
  XLC: 캐시 로드 (1894행)
  XLU: 캐시 로드 (5534행)
  XLRE: 캐시 로드 (2572행)

섹터 ETF 로드 완료: 11개
indmom 계산 완료: 11개 섹터


---
## Section 4b. SPY & ^TNX 수집

- **SPY**: Information Ratio(ir_63d) 기준 벤치마크
- **^TNX**: 미국 10년 국채 수익률 — rate_sensitivity(금리 민감도) 계산용

In [6]:
# ── SPY 수집 ─────────────────────────────────────────────────────────
if SPY_PATH.exists():
    with open(SPY_PATH, 'rb') as f:
        spy_df = pickle.load(f)
    print(f'SPY 캐시 로드 ({len(spy_df)}행)')
else:
    spy_df = safe_download('SPY')
    if spy_df is not None:
        with open(SPY_PATH, 'wb') as f:
            pickle.dump(spy_df, f)
        print(f'SPY 수집 완료 ({len(spy_df)}행)')
    else:
        print('SPY 수집 실패')
        spy_df = None

if spy_df is not None:
    spy_cols = {c.lower().replace(' ', '_'): c for c in spy_df.columns}
    _ac_col  = spy_cols.get('adj_close') or spy_cols.get('adjclose') or spy_cols.get('close')
    SPY_AC   = spy_df[_ac_col].copy()
    SPY_AC.index = SPY_AC.index.tz_localize(None) if SPY_AC.index.tz else SPY_AC.index
else:
    SPY_AC = None

# ── ^TNX (10년 국채 수익률) 수집 ─────────────────────────────────────
if DGS10_PATH.exists():
    DGS10_DAILY = pd.read_csv(DGS10_PATH, index_col='date', parse_dates=True)['close']
    print(f'^TNX 캐시 로드 ({len(DGS10_DAILY)}행)')
else:
    tnx_df = safe_download('^TNX')
    if tnx_df is not None:
        tnx_cols  = {c.lower().replace(' ', '_'): c for c in tnx_df.columns}
        _cl_col   = tnx_cols.get('close') or tnx_cols.get('adj_close')
        DGS10_DAILY = tnx_df[_cl_col].copy()
        DGS10_DAILY.index = (DGS10_DAILY.index.tz_localize(None)
                              if DGS10_DAILY.index.tz else DGS10_DAILY.index)
        DGS10_DAILY.index.name = 'date'
        DGS10_DAILY.name = 'close'
        DGS10_DAILY.to_frame().to_csv(DGS10_PATH)
        print(f'^TNX 수집 완료 ({len(DGS10_DAILY)}행)')
    else:
        DGS10_DAILY = None
        print('^TNX 수집 실패 → rate_sensitivity 미계산')

print(f'\nSPY     : {"OK" if SPY_AC      is not None else "없음"}')
print(f'DGS10   : {"OK" if DGS10_DAILY is not None else "없음"}')

SPY 캐시 로드 (5534행)
^TNX 캐시 로드 (5529행)

SPY     : OK
DGS10   : OK


---
## Section 4c. 거시/외부 지표 수집

**yfinance (7개)**: WTI원유 · 금 · 은 · VIX · 달러(DXY) · SKEW · 구리  
**FRED (6개)**: HY스프레드 · 수익률곡선(10Y-2Y) · 실업수당(주간) · Sahm · CPI · 실업률

> CPI · 실업률 · Sahm은 월간 발표 시차(~1개월) 존재 → `shift(1)` 적용으로 lookahead 방지  
> 캐시: `data/macro_yf.csv`, `data/macro_fred.csv`

In [7]:
# ── 경로 ─────────────────────────────────────────────────────────────
MACRO_YF_PATH   = DATA_DIR / 'macro_yf.csv'
MACRO_FRED_PATH = DATA_DIR / 'macro_fred.csv'

# ── yfinance 외부/대안 지표 (7개) ────────────────────────────────────
MACRO_YF_TICKERS = {
    'CL=F':     'WTI 원유 선물',
    'GC=F':     '금 선물',
    'SI=F':     '은 선물',
    '^VIX':     'VIX 공포지수',
    'DX-Y.NYB': 'DXY 달러 인덱스',
    '^SKEW':    'CBOE SKEW (꼬리위험)',
    'HG=F':     '구리 선물',
}

# 컬럼명 정규화 (특수문자 제거)
MACRO_RENAME = {
    'CL=F':         'wti_crude',
    'GC=F':         'gold',
    'SI=F':         'silver',
    '^VIX':         'vix',
    'DX-Y.NYB':     'dxy',
    '^SKEW':        'skew_idx',
    'HG=F':         'copper',
    'T10Y2Y':       't10y2y',
    'ICSA':         'icsa',
    'SAHMREALTIME': 'sahm',
    'CPIAUCSL':     'cpi',
    'UNRATE':       'unrate',
}
MACRO_COLS = list(MACRO_RENAME.values())

# ── yfinance 수집 ────────────────────────────────────────────────────
if MACRO_YF_PATH.exists():
    df_macro_yf = pd.read_csv(MACRO_YF_PATH, index_col='date', parse_dates=True)
    print(f'거시 yfinance 캐시 로드 ({df_macro_yf.shape[1]}개 컬럼, {len(df_macro_yf)}행)')
else:
    raw_yf = {}
    for sym, desc in MACRO_YF_TICKERS.items():
        df_tmp = safe_download(sym)
        if df_tmp is None:
            print(f'  {sym}: FAIL')
            continue
        cols_lower = {c.lower().replace(' ', '_'): c for c in df_tmp.columns}
        ac = cols_lower.get('adj_close') or cols_lower.get('adjclose') or cols_lower.get('close')
        if ac:
            s = df_tmp[ac].copy()
            s.index = s.index.tz_localize(None) if s.index.tz else s.index
            raw_yf[sym] = s
            print(f'  {sym}: OK ({len(s)}행) — {desc}')
        time.sleep(0.3)
    df_macro_yf = pd.DataFrame(raw_yf).rename(columns=MACRO_RENAME)
    df_macro_yf.index.name = 'date'
    df_macro_yf.to_csv(MACRO_YF_PATH, encoding='utf-8-sig')
    print(f'거시 yfinance 저장: {df_macro_yf.shape}')

# ── FRED (6개, API 키 없이 직접 CSV 다운로드) ─────────────────────────
MACRO_FRED_SERIES = {
    'T10Y2Y':       '일별',
    'ICSA':         '주간',
    'SAHMREALTIME': '월간',   # shift(1) 적용
    'CPIAUCSL':     '월간',   # shift(1) 적용
    'UNRATE':       '월간',   # shift(1) 적용
}
MONTHLY_FRED = {'SAHMREALTIME', 'CPIAUCSL', 'UNRATE'}

if MACRO_FRED_PATH.exists():
    df_macro_fred = pd.read_csv(MACRO_FRED_PATH, index_col='date', parse_dates=True)
    print(f'FRED 캐시 로드 ({df_macro_fred.shape[1]}개 컬럼, {len(df_macro_fred)}행)')
else:
    def _fetch_fred_csv(sid, start, end):
        url  = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
        r    = requests.get(url, timeout=30)
        r.raise_for_status()
        df_f = pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True)
        df_f.columns = [sid]
        return df_f.replace('.', np.nan).astype(float).loc[start:end].dropna()

    fred_raw = {}
    for sid, freq in MACRO_FRED_SERIES.items():
        try:
            s = _fetch_fred_csv(sid, PRICE_START, PRICE_END)
            fred_raw[sid] = s[sid]
            print(f'  {sid}: OK ({len(s)}행, {freq})')
        except Exception as e:
            print(f'  {sid}: FAIL — {e}')

    daily_range = pd.date_range(start=PRICE_START, end=PRICE_END, freq='D')
    fred_build  = {}
    for sid, s in fred_raw.items():
        if sid in MONTHLY_FRED:
            # 발표 시차(~1개월) 반영: 1기간 shift 후 일별 ffill
            fred_build[sid] = s.shift(1).reindex(daily_range).ffill()
        else:
            fred_build[sid] = s.reindex(daily_range).ffill()

    df_macro_fred = pd.DataFrame(fred_build).rename(columns=MACRO_RENAME)
    df_macro_fred.index.name = 'date'
    df_macro_fred.to_csv(MACRO_FRED_PATH, encoding='utf-8-sig')
    print(f'FRED 저장: {df_macro_fred.shape}')

# ── 전역 MACRO_DF ─────────────────────────────────────────────────────
MACRO_DF = pd.concat([df_macro_yf, df_macro_fred], axis=1)
MACRO_DF.index = pd.to_datetime(MACRO_DF.index)
print(f'\n전체 거시 피처: {MACRO_DF.shape[1]}개')
print(MACRO_DF.columns.tolist())

거시 yfinance 캐시 로드 (7개 컬럼, 5554행)
FRED 캐시 로드 (5개 컬럼, 8036행)

전체 거시 피처: 12개
['wti_crude', 'gold', 'silver', 'vix', 'dxy', 'skew_idx', 'copper', 't10y2y', 'icsa', 'sahm', 'cpi', 'unrate']


---
## Section 5. 개별 패널 피처 생성

**기본 피처 (김재천 파이프라인 기반)**
- 수익률: `simple_ret_1d`, `log_ret_1d`, `excess_ret_1d`
- 모멘텀: `mom_1w/1m/3m/6m/12m/12m_skip_1m`, `chmom`
- 변동성: `vol_20d/60d/252d`
- 베타/고유변동성: `beta_252d`, `idiovol_21d`
- 유동성: `dollar_vol_21d`, `amihud_21d`, `log_mcap`

**추가 피처 (논문 + 기술적 지표)**
- `ivol_63d`: 63일 고유변동성 (21일 버전과 구분)
- `low52w_ratio`: 현재가 / 52주 저점
- `ma_gap_20_60`: (MA20 − MA60) / MA60
- `rsi_14`: 14일 RSI
- `bb_pct`: Bollinger Band %B (20일, 2σ)
- `intraday_range`: (High − Low) / 전일종가 (yfinance OHLC 활용)
- `vol_surge`: 거래량 / 21일 평균 거래량
- `autocorr_21d`: 21일 수익률 자기상관
- `sharpe_21d/63d`, `sortino_63d`: rolling 리스크 조정 수익
- `ir_63d`: 63일 SPY 대비 정보비율
- `skew_63d`, `kurt_63d`: 분포 지표
- `mdd_252d`: 252일 최고점 대비 현재 하락률
- `rate_sensitivity`: 63일 excess_ret ↔ ^TNX 변화 상관계수

In [8]:
def build_panel(ticker, gics_sector, price_df, ff_df):
    df = price_df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                            .str.replace(' ', '_', regex=False))
    df = df.rename(columns={'adjclose': 'adj_close', 'stock_splits': 'split_ratio'})
    if 'adj_close' not in df.columns and 'close' in df.columns:
        df['adj_close'] = df['close']
    for col in ['split_ratio', 'dividends']:
        if col not in df.columns:
            df[col] = 0.0

    ac  = df['adj_close']
    vol = df['volume']
    r   = ac.pct_change()
    lr  = np.log(ac / ac.shift(1))
    ann = np.sqrt(252)

    df['simple_ret_1d'] = r
    df['log_ret_1d']    = lr

    # ── FF 팩터 조인 ─────────────────────────────────────────────────
    ff_cols = ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor']
    df = df.join(ff_df[[c for c in ff_cols if c in ff_df.columns]], how='left')

    rf  = df['rf']     if 'rf'     in df.columns else pd.Series(0.0,   index=df.index)
    mkt = df['mkt_rf'] if 'mkt_rf' in df.columns else pd.Series(np.nan, index=df.index)
    exc = r - rf
    df['excess_ret_1d'] = exc

    # ── 모멘텀 ──────────────────────────────────────────────────────
    r1 = 1 + r
    df['mom_1w']          = r1.rolling(5).apply(np.prod, raw=True)   - 1
    df['mom_1m']          = r1.rolling(21).apply(np.prod, raw=True)  - 1
    df['mom_3m']          = r1.rolling(63).apply(np.prod, raw=True)  - 1
    df['mom_6m']          = r1.rolling(126).apply(np.prod, raw=True) - 1
    df['mom_12m']         = r1.rolling(252).apply(np.prod, raw=True) - 1
    df['mom_12m_skip_1m'] = (r1.rolling(252).apply(np.prod, raw=True)
                             / r1.rolling(21).apply(np.prod, raw=True) - 1)
    df['chmom']           = df['mom_6m'] - df['mom_6m'].shift(126)

    # ── 변동성 ──────────────────────────────────────────────────────
    df['vol_20d']  = lr.rolling(20).std()  * ann
    df['vol_60d']  = lr.rolling(60).std()  * ann
    df['vol_252d'] = lr.rolling(252).std() * ann

    # ── 베타 & 고유변동성 ────────────────────────────────────────────
    cov_em  = exc.rolling(252).cov(mkt)
    var_m   = mkt.rolling(252).var()
    df['beta_252d']   = cov_em / var_m
    resid             = exc - df['beta_252d'] * mkt
    df['idiovol_21d'] = resid.rolling(21).std() * ann
    df['ivol_63d']    = resid.rolling(63).std() * ann

    # ── 유동성 ──────────────────────────────────────────────────────
    dollar_vol = ac * vol
    df['dollar_vol_21d'] = dollar_vol.rolling(21).mean()
    df['amihud_21d'] = (
        (r.abs() / dollar_vol.replace(0, np.nan)).rolling(21).mean() * 1e6
    )

    # ── 시가총액 ─────────────────────────────────────────────────────
    try:
        info   = yf.Ticker(ticker).fast_info
        shares = getattr(info, 'shares', None)
    except Exception:
        shares = None
    df['market_cap'] = (ac * shares) if shares else np.nan
    df['log_mcap']   = np.log(df['market_cap'].replace(0, np.nan))

    # ── 가격 패턴 ────────────────────────────────────────────────────
    df['high52w_ratio'] = ac / ac.rolling(252).max()
    df['low52w_ratio']  = ac / ac.rolling(252).min()
    df['maxret_21d']    = r.rolling(21).max()

    ma20 = ac.rolling(20).mean()
    ma60 = ac.rolling(60).mean()
    df['ma_gap_20_60'] = (ma20 - ma60) / ma60.replace(0, np.nan)

    # ── 기술적 지표 ──────────────────────────────────────────────────
    # RSI (14d)
    delta = r.copy()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))

    # Bollinger Band %B (20d, 2σ)
    std20    = ac.rolling(20).std()
    upper_bb = ma20 + 2 * std20
    lower_bb = ma20 - 2 * std20
    df['bb_pct'] = (ac - lower_bb) / (upper_bb - lower_bb).replace(0, np.nan)

    # Intraday range (High/Low 존재 시)
    if 'high' in df.columns and 'low' in df.columns:
        df['intraday_range'] = (df['high'] - df['low']) / ac.shift(1)
    else:
        df['intraday_range'] = np.nan

    # Volume surge
    df['vol_surge'] = vol / vol.rolling(21).mean().replace(0, np.nan)

    # Return autocorrelation (22d window, lag-1)
    def _autocorr1(x):
        return np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 2 else np.nan
    df['autocorr_21d'] = r.rolling(22).apply(_autocorr1, raw=True)

    # ── 리스크 조정 수익 ──────────────────────────────────────────────
    for w, name in [(21, 'sharpe_21d'), (63, 'sharpe_63d')]:
        m = exc.rolling(w).mean()
        s = exc.rolling(w).std()
        df[name] = (m / s.replace(0, np.nan)) * ann

    down_std = exc.clip(upper=0).rolling(63).std()
    df['sortino_63d'] = (exc.rolling(63).mean() / down_std.replace(0, np.nan)) * ann

    # IR (63d) vs SPY
    if SPY_AC is not None:
        spy_r  = SPY_AC.reindex(df.index).pct_change()
        active = exc - (spy_r - rf)
        m_a    = active.rolling(63).mean()
        s_a    = active.rolling(63).std()
        df['ir_63d'] = (m_a / s_a.replace(0, np.nan)) * ann
    else:
        df['ir_63d'] = np.nan

    # ── 분포 지표 ────────────────────────────────────────────────────
    df['skew_63d'] = r.rolling(63).skew()
    df['kurt_63d'] = r.rolling(63).kurt()

    # Drawdown (현재가 vs 252일 최고점)
    df['mdd_252d'] = (ac / ac.rolling(252).max()) - 1

    # ── 거시 민감도 ──────────────────────────────────────────────────
    if DGS10_DAILY is not None:
        dgs10_chg = DGS10_DAILY.reindex(df.index).diff()
        df['rate_sensitivity'] = exc.rolling(63).corr(dgs10_chg)
    else:
        df['rate_sensitivity'] = np.nan

    # ── 거시/외부 지표 조인 (Section 4c) ────────────────────────────
    if 'MACRO_DF' in globals() and MACRO_DF is not None:
        macro_aligned = MACRO_DF.reindex(df.index)
        for col in macro_aligned.columns:
            df[col] = macro_aligned[col]

    # ── 식별 컬럼 ────────────────────────────────────────────────────
    df['ticker']      = ticker
    df['gics_sector'] = gics_sector

    ordered = [
        'ticker', 'gics_sector',
        'adj_close', 'volume', 'dividends', 'split_ratio',
        'simple_ret_1d', 'log_ret_1d', 'excess_ret_1d',
        'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor',
        'mom_1w', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_skip_1m', 'chmom',
        'vol_20d', 'vol_60d', 'vol_252d',
        'beta_252d', 'idiovol_21d', 'ivol_63d',
        'dollar_vol_21d', 'amihud_21d',
        'market_cap', 'log_mcap',
        'high52w_ratio', 'low52w_ratio', 'maxret_21d', 'ma_gap_20_60',
        'rsi_14', 'bb_pct', 'intraday_range', 'vol_surge', 'autocorr_21d',
        'sharpe_21d', 'sharpe_63d', 'sortino_63d', 'ir_63d',
        'skew_63d', 'kurt_63d', 'mdd_252d',
        'rate_sensitivity',
        # ── 거시/외부 지표 (Section 4c) ─────────────────────────────
        'wti_crude', 'gold', 'silver', 'vix', 'dxy', 'skew_idx', 'copper',
        't10y2y', 'icsa', 'sahm', 'cpi', 'unrate',
    ]
    for col in ordered:
        if col not in df.columns:
            df[col] = np.nan
    return df[ordered]


# ── 루프 실행 ─────────────────────────────────────────────────────────
ok_list, fail_list = [], []

for i, ticker in enumerate(tickers, 1):
    panel_path = PANELS_DIR / f'{ticker}.csv'
    cache_path = CACHE_DIR  / f'{ticker}.pkl'

    if panel_path.exists():
        ok_list.append(ticker)
        if i % 50 == 0:
            print(f'  [{i:3d}/{len(tickers)}] 패널 스킵 중...')
        continue

    if not cache_path.exists():
        fail_list.append(ticker)
        print(f'  [{i:3d}/{len(tickers)}] {ticker}: 캐시 없음 SKIP')
        continue

    with open(cache_path, 'rb') as f:
        df_price = pickle.load(f)

    try:
        gics_sector = sector_map.get(ticker, 'Unknown')
        df_panel    = build_panel(ticker, gics_sector, df_price, df_ff)

        if gics_sector in indmom_by_sector:
            df_panel = df_panel.join(
                indmom_by_sector[gics_sector].rename('indmom'), how='left'
            )
        else:
            df_panel['indmom'] = np.nan

        df_panel.to_csv(panel_path, index=True, encoding='utf-8-sig')
        ok_list.append(ticker)
        if i % 20 == 0 or i == len(tickers):
            print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → OK ({len(df_panel)}행, '
                  f'{df_panel.shape[1]}피처)')
    except Exception as e:
        fail_list.append(ticker)
        print(f'  [{i:3d}/{len(tickers)}] {ticker}: FAIL — {e}')

print(f'\n완료: 성공 {len(ok_list)}개 / 실패 {len(fail_list)}개')
if fail_list:
    print(f'실패: {fail_list}')

  [ 50/498] 패널 스킵 중...
  [100/498] 패널 스킵 중...
  [150/498] 패널 스킵 중...
  [200/498] 패널 스킵 중...
  [250/498] 패널 스킵 중...
  [300/498] 패널 스킵 중...
  [350/498] 패널 스킵 중...
  [400/498] 패널 스킵 중...
  [450/498] 패널 스킵 중...

완료: 성공 498개 / 실패 0개


---
## Section 6. 월별 패널 생성

**처리 순서**
1. 모든 개별 패널 로드 + indmom 조인 + 종속변수(`fwd_excess_ret_1m`) 계산
2. 21거래일 단위 서브샘플링 (레이블 중복 제거)
3. **동적 유니버스 필터**: 각 날짜의 실제 S&P 500 구성 종목만 유지
4. **`avg_corr`** 계산: 63일 롤링 페어와이즈 상관계수 평균
5. **횡단면 랭킹**: `ret_rank_1m`, `ret_rank_3m`, `vol_rank`

$$r^{excess}_{t \to t+21} = \prod_{k=1}^{21}(1 + r^{simple}_{t+k}) - 1 - rf_{t \to t+21}$$

In [9]:
# ── 패널 로드 ─────────────────────────────────────────────────────────
all_panels     = []
daily_ret_dict = {}

for ticker in tickers:
    panel_path = PANELS_DIR / f'{ticker}.csv'
    if not panel_path.exists():
        continue

    df = pd.read_csv(panel_path, index_col='date', parse_dates=True)

    gics_sector = sector_map.get(ticker, 'Unknown')
    if gics_sector in indmom_by_sector and 'indmom' not in df.columns:
        df = df.join(indmom_by_sector[gics_sector].rename('indmom'), how='left')
    elif 'indmom' not in df.columns:
        df['indmom'] = np.nan

    # 종속변수: 21일 선행 초과수익률
    r1          = 1 + df['simple_ret_1d']
    fwd_21      = r1.shift(-1).rolling(21).apply(np.prod, raw=True).shift(-20) - 1
    rf1         = 1 + df['rf'].fillna(0)
    fwd_rf_21   = rf1.shift(-1).rolling(21).apply(np.prod, raw=True).shift(-20) - 1
    df['fwd_excess_ret_1m'] = fwd_21 - fwd_rf_21

    all_panels.append(df)
    daily_ret_dict[ticker] = df['simple_ret_1d']

print(f'로드된 패널: {len(all_panels)}종목')
combined = pd.concat(all_panels, axis=0).sort_index()
combined.index = pd.to_datetime(combined.index)

# ── 월말 서브샘플링 (GKX 방식: 각 종목의 월말 마지막 관측값) ────────────
# iloc[::21] 대신 resample('ME').last() 사용
# → 모든 종목이 동일한 월말 날짜에 정렬되어 cross-section rank가 의미를 가짐
monthly_panels = []
for ticker in combined['ticker'].unique():
    tk_df = combined[combined['ticker'] == ticker].copy()
    tk_df = tk_df.sort_index()
    tk_monthly = (
        tk_df.resample('ME')
        .last()
        .dropna(subset=['fwd_excess_ret_1m'])
    )
    # resample 후 ticker/gics_sector가 NaN이 될 수 있으므로 복원
    tk_monthly['ticker']      = ticker
    tk_monthly['gics_sector'] = sector_map.get(ticker, 'Unknown')
    monthly_panels.append(tk_monthly)

monthly_df = pd.concat(monthly_panels, axis=0).sort_index()
monthly_df.index.name = 'date'
print(f'월말 서브샘플 후: {monthly_df.shape}')
print(f'날짜당 종목 수 (샘플):')
counts = monthly_df.groupby(level='date')['ticker'].count()
print(f'  평균: {counts.mean():.0f}  중앙값: {counts.median():.0f}  최솟값: {counts.min()}  최댓값: {counts.max()}')

# ── 동적 유니버스 필터 ────────────────────────────────────────────────
print('동적 유니버스 필터 적용 중...')
before   = len(monthly_df)
filtered = []
for date, group in monthly_df.groupby(level='date'):
    members = get_sp500_members_at(date)
    filtered.append(group[group['ticker'].isin(members)])
monthly_df = pd.concat(filtered)
monthly_df.index.name = 'date'
print(f'  {before:,}행 → {len(monthly_df):,}행 (동적 필터 후)')
print(f'날짜당 종목 수 (필터 후):')
counts2 = monthly_df.groupby(level='date')['ticker'].count()
print(f'  평균: {counts2.mean():.0f}  중앙값: {counts2.median():.0f}  최솟값: {counts2.min()}  최댓값: {counts2.max()}')

# ── avg_corr (63거래일 rolling 페어와이즈 상관계수 평균) ──────────────
print('avg_corr 계산 중 (잠시 시간이 걸릴 수 있음)...')
daily_ret_df   = pd.DataFrame(daily_ret_dict)
avg_corr_recs  = []

for date in sorted(monthly_df.index.unique()):
    window = daily_ret_df.loc[date - pd.Timedelta(days=90): date].dropna(how='all', axis=1)
    if window.shape[1] < 5 or len(window) < 21:
        continue
    corr_m = window.corr()
    for t in corr_m.columns:
        others = corr_m[t].drop(t, errors='ignore')
        avg_corr_recs.append({'date': date, 'ticker': t, 'avg_corr': others.mean()})

if avg_corr_recs:
    avg_corr_ser = pd.DataFrame(avg_corr_recs).set_index(['date', 'ticker'])['avg_corr']
    monthly_df   = (monthly_df.reset_index()
                    .merge(avg_corr_ser.reset_index(), on=['date', 'ticker'], how='left')
                    .set_index('date'))
else:
    monthly_df['avg_corr'] = np.nan

# ── 횡단면 랭킹 (날짜별 백분위) ──────────────────────────────────────
monthly_df['ret_rank_1m'] = monthly_df.groupby(level='date')['mom_1m'].rank(pct=True)
monthly_df['ret_rank_3m'] = monthly_df.groupby(level='date')['mom_3m'].rank(pct=True)
monthly_df['vol_rank']    = monthly_df.groupby(level='date')['vol_20d'].rank(pct=True)
monthly_df.index.name = 'date'

print(f'\n월별 패널: {monthly_df.shape}')
print(f'  종목 수: {monthly_df["ticker"].nunique()}')
print(f'  날짜 수: {monthly_df.index.nunique()}')
print(f'  기간: {monthly_df.index.min().date()} ~ {monthly_df.index.max().date()}')
print(f'\nfwd_excess_ret_1m 분포:')
print(monthly_df['fwd_excess_ret_1m'].describe())

로드된 패널: 498종목
월말 서브샘플 후: (118431, 65)
날짜당 종목 수 (샘플):
  평균: 450  중앙값: 459  최솟값: 375  최댓값: 498
동적 유니버스 필터 적용 중...
  118,431행 → 93,113행 (동적 필터 후)
날짜당 종목 수 (필터 후):
  평균: 354  중앙값: 332  최솟값: 271  최댓값: 493
avg_corr 계산 중 (잠시 시간이 걸릴 수 있음)...

월별 패널: (93113, 69)
  종목 수: 494
  날짜 수: 263
  기간: 2004-01-31 ~ 2025-11-30

fwd_excess_ret_1m 분포:
count    93113.000000
mean         0.010707
std          0.086865
min         -0.845578
25%         -0.035531
50%          0.011351
75%          0.056437
max          2.449770
Name: fwd_excess_ret_1m, dtype: float64


---
## Section 7. 저장 및 요약

In [10]:
# 저장
OUT_PATH = DATA_DIR / 'monthly_panel.csv'
monthly_df.to_csv(OUT_PATH, encoding='utf-8-sig')
print(f'저장 완료: {OUT_PATH}')
print(f'파일 크기: {OUT_PATH.stat().st_size / 1024 / 1024:.1f} MB')

# ── 요약 통계 ─────────────────────────────────────────────────────────
print('\n=== 피처 NaN 비율 (상위 10개) ===')
nan_rate = monthly_df.isnull().mean().sort_values(ascending=False)
print(nan_rate.head(10).to_string())

print('\n=== 섹터별 종목 수 ===')
print(monthly_df.groupby('gics_sector')['ticker'].nunique().to_string())

print('\n=== 전체 컬럼 목록 ===')
print(monthly_df.columns.tolist())

print('\n=== 샘플 (첫 3행) ===')
monthly_df.head(3)

저장 완료: c:\workspace\camp\project\finance_project\서윤범\Step3_IndividualStocks\data\monthly_panel.csv
파일 크기: 96.1 MB

=== 피처 NaN 비율 (상위 10개) ===
hy_spread           0.837713
indmom              0.077959
ivol_63d            0.049585
idiovol_21d         0.042959
chmom               0.039651
vol_252d            0.039651
beta_252d           0.039651
mom_12m             0.039651
mom_12m_skip_1m     0.039651
rate_sensitivity    0.038867

=== 섹터별 종목 수 ===
gics_sector
Communication Services    22
Consumer Discretionary    47
Consumer Staples          35
Energy                    22
Financials                75
Health Care               58
Industrials               77
Information Technology    71
Materials                 25
Real Estate               31
Utilities                 31

=== 전체 컬럼 목록 ===
['ticker', 'gics_sector', 'adj_close', 'volume', 'dividends', 'split_ratio', 'simple_ret_1d', 'log_ret_1d', 'excess_ret_1d', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor', 'mom_1w', 'mom_1m'

,ticker,gics_sector,adj_close,volume,dividends,split_ratio,simple_ret_1d,log_ret_1d,excess_ret_1d,mkt_rf,...,icsa,sahm,cpi,unrate,indmom,fwd_excess_ret_1m,avg_corr,ret_rank_1m,ret_rank_3m,vol_rank
date,,,,,,,,,,,,,,,,,,,,,
2004-01-31,A,Health Care,21.982086,7774418,0.0,0.0,0.033941,0.033378,0.033941,-0.0015,...,353000.0,NaN,NaN,NaN,NaN,-0.071351,NaN,NaN,NaN,NaN
2004-01-31,HSY,Consumer Staples,22.230761,1726400,0.0,0.0,0.001459,0.001458,0.001459,-0.0015,...,353000.0,NaN,NaN,NaN,NaN,0.102274,NaN,NaN,NaN,NaN
2004-01-31,AMGN,Health Care,43.735298,6730800,0.0,0.0,-0.005077,-0.005090,-0.005077,-0.0015,...,353000.0,NaN,NaN,NaN,NaN,-0.017322,NaN,NaN,NaN,NaN


---
## 완료 요약

| 항목 | 내용 |
|---|---|
| 유니버스 | S&P 500 동적 구성 (2004~2024 기간 중 편입 종목 전체, GICS 섹터 보유 종목) |
| 생존편향 | Wikipedia 변경 히스토리 역방향 재구성으로 제거 |
| 수집 기간 | 2004-01-01 ~ 2024-12-31 (20년) |
| 피처 수 | ~50개 (수익률 · 모멘텀 · 변동성 · 리스크 · 유동성 · 기술적 · 거시) |
| 횡단면 피처 | ret_rank_1m, ret_rank_3m, vol_rank, avg_corr (월별 패널 단계에서 계산) |
| 종속변수 | `fwd_excess_ret_1m`: 1개월 선행 초과수익률 |
| 산출물 | `data/monthly_panel.csv` |

**다음 단계**: `02_XGBoost_WalkForward.ipynb`
- 누적 확장 윈도우(expanding IS) + OOS = 1개월
- XGBRegressor (Huber loss) + Optuna 하이퍼파라미터 탐색
- 개별 피처 횡단면 랭크 정규화 → [-1, 1] / 거시 피처 rolling z-score
- R²_OOS (benchmark=0), IC 분석
- Q (예측 수익률), Ω (불확실성) 생성 → Black-Litterman 입력